In [ ]:
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [ ]:
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return response.json()


def get_movie_credits(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}/credits")
    return response.json()

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie.",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew credits for a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie.",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [ ]:
available_functions = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}


def ask_agent(user_message):
    print(f"User: {user_message}\n")

    messages = [
        {
            "role": "system",
            "content": "You are a movie expert assistant. Use the provided functions to look up movie information and answer user questions in Korean.",
        },
        {"role": "user", "content": user_message},
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )

    message = response.choices[0].message

    if message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(f"Calling: {function_name}({function_args})")

            fn = available_functions[function_name]
            result = fn(**function_args)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )

        answer = final_response.choices[0].message.content
    else:
        answer = message.content

    print(f"\nAssistant: {answer}")
    return answer

In [ ]:
ask_agent("지금 인기 있는 영화가 무엇인지 알려줘")

User: 지금 인기 있는 영화가 무엇인지 알려줘



Calling: get_popular_movies({})



Assistant: 현재 인기 있는 영화들은 다음과 같습니다:

1. **너의 심장은 부서질 것이다 (Your Heart Will Be Broken)**
   - 개요: 고등학생 폴리나가 새 학교에서 괴롭힘을 당하다가, 주 괴롭히는 바르스와의 거래로 그의 남자친구인 척 해주기로 합니다. 이 게임을 통해 진정한 감정이 생기지만, 그녀의 가족과 친구들은 이 커플을 헤어지게 하려 합니다.
   - 개봉일: 2026년 3월 26일
   - 평점: 6.5
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **아바타: 불과 재 (Avatar: Fire and Ash)**
   - 개요: RDA와의 전쟁 후, 제이크 썰리와 네이티리가 판도라에서 새로운 위협인 재 사람들로부터 가족을 지키기 위해 싸웁니다.
   - 개봉일: 2025년 12월 17일
   - 평점: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **호퍼스 (Hoppers)**
   - 개요: 과학자들이 인간의 의식을 생생한 로봇 동물에 '홉' 할 수 있는 방법을 발견하고, 동물과 소통할 수 있게 됩니다. 동물 애호가 메이벨은 이 기술을 사용하여 동물 세계의 비밀을 밝혀냅니다.
   - 개봉일: 2026년 3월 4일
   - 평점: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **범죄 101 (Crime 101)**
   - 개요: 고위험 목표를 설정한 은밀한 도둑과 그의 마지막 작전을 놓고 조사를 벌이는 보험 중개인, 그리고 이 사건을 해결하고자 하는 집요한 탐정의 이야기가 펼쳐집니다.
   - 개봉일: 2026년 2월 11일
   - 평점: 7.1
   - ![포스터](https://image.tmdb.

"현재 인기 있는 영화들은 다음과 같습니다:\n\n1. **너의 심장은 부서질 것이다 (Your Heart Will Be Broken)**\n   - 개요: 고등학생 폴리나가 새 학교에서 괴롭힘을 당하다가, 주 괴롭히는 바르스와의 거래로 그의 남자친구인 척 해주기로 합니다. 이 게임을 통해 진정한 감정이 생기지만, 그녀의 가족과 친구들은 이 커플을 헤어지게 하려 합니다.\n   - 개봉일: 2026년 3월 26일\n   - 평점: 6.5\n   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)\n\n2. **아바타: 불과 재 (Avatar: Fire and Ash)**\n   - 개요: RDA와의 전쟁 후, 제이크 썰리와 네이티리가 판도라에서 새로운 위협인 재 사람들로부터 가족을 지키기 위해 싸웁니다.\n   - 개봉일: 2025년 12월 17일\n   - 평점: 7.4\n   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)\n\n3. **호퍼스 (Hoppers)**\n   - 개요: 과학자들이 인간의 의식을 생생한 로봇 동물에 '홉' 할 수 있는 방법을 발견하고, 동물과 소통할 수 있게 됩니다. 동물 애호가 메이벨은 이 기술을 사용하여 동물 세계의 비밀을 밝혀냅니다.\n   - 개봉일: 2026년 3월 4일\n   - 평점: 7.6\n   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)\n\n4. **범죄 101 (Crime 101)**\n   - 개요: 고위험 목표를 설정한 은밀한 도둑과 그의 마지막 작전을 놓고 조사를 벌이는 보험 중개인, 그리고 이 사건을 해결하고자 하는 집요한 탐정의 이야기가 펼쳐집니다.\n   - 개봉일: 2026년 2월 11일\n   - 평점: 7.1\n   - ![포스터](https:

In [ ]:
ask_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")

User: movie ID 550에 해당하는 영화가 무엇인지 알려줘



Calling: get_movie_details({'movie_id': 550})



Assistant: 영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 이 영화는 1999년 10월 15일에 개봉하였으며, 장르는 드라마와 스릴러입니다. 

**줄거리:** 불면증에 시달리는 남자와 미끄러운 비누 판매자가 원초적인 남성 공격성을 새로운 형태의 치료로 Channel 합니다. 이들의 개념은 모든 도시에 '격투 클럽'이 형성되면서 인기를 끌고, 이 과정에서 한 괴짜가 등장하여 제어할 수 없는 혼돈으로 이어지게 됩니다.

- **예산:** 63,000,000 달러
- **수익:** 100,853,753 달러
- **상영 시간:** 139분
- **평점:** 8.439 (31728 투표 기준)
- **홈페이지:** [Fight Club](https://www.20thcenturystudios.com/movies/fight-club)

![포스터](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)

**태그라인:** "Mischief. Mayhem. Soap."


'영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 이 영화는 1999년 10월 15일에 개봉하였으며, 장르는 드라마와 스릴러입니다. \n\n**줄거리:** 불면증에 시달리는 남자와 미끄러운 비누 판매자가 원초적인 남성 공격성을 새로운 형태의 치료로 Channel 합니다. 이들의 개념은 모든 도시에 \'격투 클럽\'이 형성되면서 인기를 끌고, 이 과정에서 한 괴짜가 등장하여 제어할 수 없는 혼돈으로 이어지게 됩니다.\n\n- **예산:** 63,000,000 달러\n- **수익:** 100,853,753 달러\n- **상영 시간:** 139분\n- **평점:** 8.439 (31728 투표 기준)\n- **홈페이지:** [Fight Club](https://www.20thcenturystudios.com/movies/fight-club)\n\n![포스터](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)\n\n**태그라인:** "Mischief. Mayhem. Soap."'

In [ ]:
ask_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")

User: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘



Calling: get_movie_credits({'movie_id': 550})



Assistant: 영화 ID 550에 해당하는 영화의 출연진은 다음과 같습니다:

1. **에드워드 노튼 (Edward Norton)** - 역할: 내레이터 (Narrator)  
   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **브래드 피트 (Brad Pitt)** - 역할: 타일러 더든 (Tyler Durden)  
   ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)

3. **헬레나 본햄 카터 (Helena Bonham Carter)** - 역할: 말라 싱어 (Marla Singer)  
   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **미트 로프 (Meat Loaf)** - 역할: 로버트 폴슨 (Robert Paulson)  
   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **자레드 레토 (Jared Leto)** - 역할: 엔젤 페이스 (Angel Face)  
   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

이 외에도 여러 인물들이 출연했습니다. 궁금한 점이 더 있으면 말씀해 주세요!


'영화 ID 550에 해당하는 영화의 출연진은 다음과 같습니다:\n\n1. **에드워드 노튼 (Edward Norton)** - 역할: 내레이터 (Narrator)  \n   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)\n\n2. **브래드 피트 (Brad Pitt)** - 역할: 타일러 더든 (Tyler Durden)  \n   ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)\n\n3. **헬레나 본햄 카터 (Helena Bonham Carter)** - 역할: 말라 싱어 (Marla Singer)  \n   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)\n\n4. **미트 로프 (Meat Loaf)** - 역할: 로버트 폴슨 (Robert Paulson)  \n   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)\n\n5. **자레드 레토 (Jared Leto)** - 역할: 엔젤 페이스 (Angel Face)  \n   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)\n\n이 외에도 여러 인물들이 출연했습니다. 궁금한 점이 더 있으면 말씀해 주세요!'